# 03. Guardrails and Human in the Loop

This notebook is a focused guardrails lesson. It shows how a guardrail checks whether the input is safe to process, and how the workflow stops or routes to a person when the text is uncertain.

## Why this matters in DH
In digital humanities, uncertainty is not always a failure. It can be part of the evidence. A blurred place name, a conflicting date, or a doubtful transcription often needs scholarly judgment.

The goal is not to eliminate ambiguity. The goal is to make ambiguity visible and actionable.

## Concepts
- Guardrails
- Human review
- Pass/fail checkpoints
- Validation before action
- Evidence-based decision making

Relevant docs: [Guardrails](https://openai.github.io/openai-agents-python/guardrails/) and [Human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/).

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
The notebook uses a small set of ambiguous archival-style records stored in `data/` and loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Human in the loop: https://openai.github.io/openai-agents-python/human_in_the_loop/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

records = [{'id': item['id'], 'text': item['text']} for item in load_documents()]
records

In [ ]:
@dataclass
class ReviewDecision:
    record_id: int
    uncertain: bool
    confidence: float
    notes: str


## Step 1: A tiny review heuristic

We treat low confidence or uncertain OCR as a signal for human review. This helper is the policy: it decides whether a record should be reviewed.

In [ ]:
def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

clean_decision = ReviewDecision(record_id=1, uncertain=False, confidence=0.95, notes='clear transcription')
uncertain_decision = ReviewDecision(record_id=3, uncertain=True, confidence=0.52, notes='OCR ambiguity around place name')
print('clean ->', should_review(clean_decision))
print('uncertain ->', should_review(uncertain_decision))

## Step 2: Build a guardrail

A guardrail can reject inputs that clearly need human review before the agent continues. Here, `should_review()` holds the policy, and a small text-to-decision helper applies that policy through the SDK.

You can think of them as automated quality checks that sit in front of or behind the agent.

In [ ]:
def review_decision_from_text(input_text: str) -> ReviewDecision:
    if not isinstance(input_text, str):
        return ReviewDecision(record_id=0, uncertain=True, confidence=0.0, notes='input is not text')

    text = input_text.lower()
    uncertainty_terms = ['may be', 'unclear', 'uncertain', 'blurred', 'partially legible']

    uncertainty_hits = sum(term in text for term in uncertainty_terms)

    suspicious_chars = '^~`_|/[]{}<>'
    suspicious_char_hits = sum(ch in suspicious_chars for ch in input_text)
    digit_letter_switch_hits = sum(
        (input_text[i].isalpha() and input_text[i + 1].isdigit())
        or (input_text[i].isdigit() and input_text[i + 1].isalpha())
        for i in range(len(input_text) - 1)
    )
    replacement_char_hits = input_text.count('\ufffd')

    ocr_hits = 0
    if suspicious_char_hits >= 2:
        ocr_hits += 1
    if digit_letter_switch_hits >= 2:
        ocr_hits += 1
    if replacement_char_hits >= 1:
        ocr_hits += 1

    penalty = 0.2 * uncertainty_hits + 0.1 * ocr_hits
    confidence = max(0.0, min(1.0, 1.0 - penalty))
    uncertain = uncertainty_hits > 0 or ocr_hits >= 2

    notes = []
    if uncertainty_hits:
        notes.append('explicit uncertainty language detected')
    if ocr_hits:
        notes.append('generic ocr-noise signals detected')
    if not notes:
        notes.append('clear enough to process')

    return ReviewDecision(
        record_id=0,
        uncertain=uncertain,
        confidence=round(confidence, 2),
        notes='; '.join(notes),
    )

def uncertainty_guardrail(ctx, agent, input_text):
    decision = review_decision_from_text(input_text)
    if should_review(decision):
        return GuardrailFunctionOutput(output_info=decision.notes, tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
guardrail

## Step 3: Agent with guardrail

If the text looks uncertain, the agent should stop and let the person review it. This is the main classroom lesson: the guardrail turns the review policy into a runtime stop signal.

For research settings, that protects against fabricated precision and makes the workflow more trustworthy.

In [ ]:
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
    model=DEFAULT_MODEL,
)

review_agent

## Step 4: Use one process for both cases

Run both records through the same helper. If the guardrail passes, the run continues normally. If it triggers, we show a simulated human-review step.

In [ ]:
def simulate_human_review(record: dict, decision: ReviewDecision) -> str:
    return (
        f"Human review for record {record['id']}: verify transcription and approve before resume. "
        f"Reason: {decision.notes}. Confidence: {decision.confidence}"
    )

async def run_case(label: str, record: dict):
    text = record['text']
    decision = review_decision_from_text(text)
    print(f"\n--- {label} (record {record['id']}) ---")
    print('Policy decision:', decision)
    try:
        with trace(f'DH guardrails {label}'):
            result = await Runner.run(review_agent, text)
        print('Pipeline output:', result.final_output)
    except Exception as exc:
        print('Guardrail triggered:', type(exc).__name__, exc)
        print(simulate_human_review(record, decision))

await run_case('clean case', records[0])

## Step 5: Same process on uncertain text

Now run an ambiguous record through the exact same helper. The only difference should be the guardrail outcome and the handoff to human review.

In [ ]:
await run_case('uncertain case', records[-1])

## Human-in-the-loop checkpoint

If the guardrail triggers, the next step is not to force a guess. It is to ask a person to resolve the ambiguity and then resume.

## Exercise

Change the threshold so only records with both OCR ambiguity and low confidence are sent to review.

## Solution

Modify `uncertainty_guardrail` to check for two conditions at once, for example `('unclear' in text and confidence < 0.8)`.